# Batch COMET-STOmics Alignment: Endo vs MLA

Batch processing notebook that runs the COMET-STOmics alignment pipeline on all available
Endometrioid (Endo) and MLA disease samples.

**Prerequisites:**
- Validate pipeline on SO34 pilot (script03_comet_stomics_alignment.ipynb)
- Export missing GeoJSON for SO58/SO59 (script04_export_missing_geojson.py)
- Locate STOmics cellbin data for remaining samples

**Outputs per sample:**
- Warped GeoJSON (COMET cells in STOmics space)
- Integrated AnnData (COMET cells x genes)
- Cell mapping CSV (COMET <-> STOmics)
- Alignment metrics JSON
- Validation plots

---

In [ ]:
import sys
import json
import logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_DIR = Path(r'T:/0_Organizational/Git/MDACC-STOmics-COMET-MSI')
sys.path.insert(0, str(REPO_DIR))

from comet.alignment_utils import (
    DefaultPaths, ENDO_SAMPLES, MLA_SAMPLES, ALL_ENDO_MLA,
    ALL_SAMPLES, OVARIAN_WITH_STOMICS, SAMPLES_WITH_CELLBIN,
    check_sample_data, run_alignment_pipeline,
    plot_alignment_validation, plot_distance_distribution,
    validate_protein_gene_correlation, PROTEIN_GENE_MAP,
)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)

plt.rcParams['figure.figsize'] = [12, 8]
plt.rcParams['figure.dpi'] = 100

print('Imports OK')

In [ ]:
# ============= CONFIGURATION =============

OUTPUT_DIR = Path('T:/Sammy Data/projects/out/comet_stomics_alignment')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pipeline parameters
AGGREGATION_METHOD = 'nearest'
MAX_DISTANCE = 50  # pixels

# Which samples to process
# Options: 'endo_mla' (default), 'endo_only', 'mla_only', 'all_with_data',
#          'all_samples' (all 53), 'custom'
SAMPLE_SET = 'endo_mla'

# For custom selection:
# CUSTOM_SAMPLES = ['SO34', 'SO4']

# Whether to skip already-processed samples
SKIP_EXISTING = True

# Whether to save registered slide images (large files)
SAVE_REGISTERED_SLIDES = False

print(f'Output: {OUTPUT_DIR}')
print(f'Sample set: {SAMPLE_SET}')
print(f'Skip existing: {SKIP_EXISTING}')

## 1. Data Availability Check

In [ ]:
# Build sample list based on configuration
if SAMPLE_SET == 'endo_mla':
    target_samples = ALL_ENDO_MLA
elif SAMPLE_SET == 'endo_only':
    target_samples = ENDO_SAMPLES
elif SAMPLE_SET == 'mla_only':
    target_samples = MLA_SAMPLES
elif SAMPLE_SET == 'all_with_data':
    target_samples = {**ALL_ENDO_MLA, **OVARIAN_WITH_STOMICS}
elif SAMPLE_SET == 'all_samples':
    target_samples = ALL_SAMPLES
elif SAMPLE_SET == 'custom':
    target_samples = {s: ALL_SAMPLES.get(s, {})
                      for s in CUSTOM_SAMPLES if s in ALL_SAMPLES}
else:
    raise ValueError(f'Unknown sample set: {SAMPLE_SET}')

# Check data availability for all target samples
all_status = []
for sid, info in target_samples.items():
    status = check_sample_data(sid, info['chip_id'], info.get('aligned', False))
    status['disease'] = info.get('disease', 'Unknown')
    
    # Determine pipeline readiness
    status['can_register'] = status['geojson'] and status['comet_bs'] and status['stomics_dapi']
    status['can_integrate'] = status['can_register'] and (status['stomics_cellbin'] or status['stomics_h5ad'])
    status['can_warp_only'] = status['geojson'] and status['comet_bs'] and status['stomics_dapi']
    
    all_status.append(status)

status_df = pd.DataFrame(all_status)

print(f'Target samples: {len(target_samples)}')
print(f'Can run full pipeline (register + integrate): {status_df["can_integrate"].sum()}')
print(f'Can register + warp only (no STOmics expr): {status_df["can_warp_only"].sum()}')
print(f'Cannot process: {(~status_df["can_warp_only"]).sum()}')

print('\nDetailed Status:')
print('=' * 100)
cols = ['sample_id', 'chip_id', 'disease', 'geojson', 'comet_bs',
        'stomics_dapi', 'stomics_cellbin', 'can_integrate']
print(status_df[cols].to_string(index=False))

In [ ]:
# Build the list of samples to process
samples_to_run = []

for _, row in status_df.iterrows():
    sid = row['sample_id']
    chip = row['chip_id']
    
    if not row['can_warp_only']:
        print(f'SKIP {sid}: missing required data')
        continue
    
    if SKIP_EXISTING:
        result_dir = OUTPUT_DIR / f'{sid}_{chip}'
        metrics_file = result_dir / f'{sid}_{chip}_alignment_metrics.json'
        if metrics_file.exists():
            print(f'SKIP {sid}: already processed')
            continue
    
    samples_to_run.append({
        'sample_id': sid,
        'chip_id': chip,
        'disease': row['disease'],
        'has_stomics': row['stomics_cellbin'] or row['stomics_h5ad'],
    })

print(f'\nSamples to process: {len(samples_to_run)}')
for s in samples_to_run:
    stomics_str = 'with STOmics' if s['has_stomics'] else 'WARP ONLY'
    print(f'  {s["sample_id"]} ({s["disease"]}, {s["chip_id"]}) - {stomics_str}')

## 2. Batch Processing

In [ ]:
# Run pipeline for each sample
batch_results = []
start_time = datetime.now()

for i, sample in enumerate(samples_to_run):
    sid = sample['sample_id']
    chip = sample['chip_id']
    
    print(f'\n{"#"*70}')
    print(f'Processing {i+1}/{len(samples_to_run)}: {sid} ({sample["disease"]}, {chip})')
    print(f'{"#"*70}')
    
    try:
        result = run_alignment_pipeline(
            sample_id=sid,
            chip_id=chip,
            work_dir=OUTPUT_DIR,
            aggregation_method=AGGREGATION_METHOD,
            max_distance=MAX_DISTANCE,
            save_registered_slides=SAVE_REGISTERED_SLIDES,
        )
        
        # Extract key metrics for summary
        summary = {
            'sample_id': sid,
            'chip_id': chip,
            'disease': sample['disease'],
            'status': result.get('status', 'unknown'),
            'n_comet_cells': result.get('n_comet_cells', 0),
        }
        
        if 'alignment_metrics' in result:
            m = result['alignment_metrics']
            summary.update({
                'quality': m['quality'],
                'median_distance': m['median_distance'],
                'pct_within_50px': m['pct_within_50px'],
                'n_stomics_cells': m['n_stomics_cells'],
            })
        
        if 'integrated_adata' in result:
            summary['n_integrated_genes'] = result['integrated_adata'].n_vars
        
        batch_results.append(summary)
        
    except Exception as e:
        print(f'ERROR processing {sid}: {e}')
        import traceback
        traceback.print_exc()
        batch_results.append({
            'sample_id': sid,
            'chip_id': chip,
            'disease': sample['disease'],
            'status': f'error: {str(e)[:50]}',
        })

elapsed = datetime.now() - start_time
print(f'\nBatch processing complete in {elapsed}')

## 3. Batch Results Summary

In [ ]:
# Summary table
results_df = pd.DataFrame(batch_results)

print('=' * 90)
print('BATCH PROCESSING RESULTS')
print('=' * 90)
print(results_df.to_string(index=False))
print('=' * 90)

# Success rate
n_complete = (results_df['status'] == 'complete').sum()
print(f'\nComplete: {n_complete}/{len(results_df)}')

# Save summary
summary_path = OUTPUT_DIR / 'batch_summary.csv'
results_df.to_csv(summary_path, index=False)
print(f'Saved summary: {summary_path}')

In [ ]:
# Compare alignment quality across samples
if 'median_distance' in results_df.columns:
    completed = results_df[results_df['status'] == 'complete'].copy()
    
    if len(completed) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Bar chart of median distances
        ax = axes[0]
        colors = completed['disease'].map({'Endo': 'steelblue', 'MLA': 'coral', 'Ovarian': 'green'})
        ax.barh(completed['sample_id'], completed['median_distance'], color=colors)
        ax.axvline(30, color='green', linestyle=':', alpha=0.7, label='30px (excellent)')
        ax.axvline(50, color='orange', linestyle=':', alpha=0.7, label='50px (good)')
        ax.set_xlabel('Median Distance (pixels)')
        ax.set_title('Alignment Quality by Sample')
        ax.legend()
        
        # Box plot by disease group
        ax = axes[1]
        disease_groups = completed.groupby('disease')['median_distance']
        for disease, group in disease_groups:
            color = {'Endo': 'steelblue', 'MLA': 'coral', 'Ovarian': 'green'}.get(disease, 'gray')
            ax.scatter([disease] * len(group), group, c=color, s=100, alpha=0.7, edgecolors='black')
        ax.set_ylabel('Median Distance (pixels)')
        ax.set_title('Alignment Quality by Disease Group')
        ax.axhline(30, color='green', linestyle=':', alpha=0.7)
        ax.axhline(50, color='orange', linestyle=':', alpha=0.7)
        
        plt.tight_layout()
        plt.savefig(str(OUTPUT_DIR / 'batch_alignment_quality.png'), dpi=150)
        plt.show()

## 4. Load All Processed Results

Load integrated AnnData objects from all successfully processed samples for combined analysis.

In [ ]:
import anndata as ad

# Scan output directory for completed samples
# Search across all known samples, not just Endo/MLA
search_samples = ALL_SAMPLES if SAMPLE_SET == 'all_samples' else ALL_ENDO_MLA

all_adatas = []
all_metrics = []

for sid, info in search_samples.items():
    chip = info['chip_id']
    sample_dir = OUTPUT_DIR / f'{sid}_{chip}'
    
    h5ad_path = sample_dir / f'{sid}_{chip}_comet_stomics_integrated.h5ad'
    metrics_path = sample_dir / f'{sid}_{chip}_alignment_metrics.json'
    
    if h5ad_path.exists():
        adata = ad.read_h5ad(str(h5ad_path))
        adata.obs['sample_id'] = sid
        adata.obs['disease'] = info['disease']
        all_adatas.append(adata)
        print(f'Loaded {sid} ({info["disease"]}): {adata.shape}')
    
    if metrics_path.exists():
        with open(metrics_path) as f:
            m = json.load(f)
        m['sample_id'] = sid
        m['disease'] = info['disease']
        all_metrics.append(m)

print(f'\nLoaded {len(all_adatas)} integrated AnnData objects')
print(f'Loaded {len(all_metrics)} alignment metrics')

In [ ]:
# Combine all AnnData objects
if len(all_adatas) > 1:
    # Find common genes across all samples
    common_genes = set(all_adatas[0].var_names)
    for adata in all_adatas[1:]:
        common_genes &= set(adata.var_names)
    common_genes = sorted(common_genes)
    
    print(f'Common genes across samples: {len(common_genes)}')
    
    # Subset to common genes and concatenate
    adatas_common = [adata[:, common_genes].copy() for adata in all_adatas]
    combined = ad.concat(adatas_common, join='inner', merge='same')
    
    print(f'Combined AnnData: {combined.shape}')
    print(f'\nSamples per disease:')
    print(combined.obs['disease'].value_counts())
    
    # Save combined
    combined_path = OUTPUT_DIR / 'combined_endo_mla_comet_stomics.h5ad'
    combined.write_h5ad(str(combined_path))
    print(f'\nSaved combined: {combined_path}')

elif len(all_adatas) == 1:
    combined = all_adatas[0]
    print(f'Only one sample loaded: {combined.shape}')
else:
    print('No AnnData objects to combine')
    combined = None

## 5. Endo vs MLA Preliminary Comparison

Quick differential analysis between disease groups (when enough samples are processed).

In [ ]:
if combined is not None and combined.obs['disease'].nunique() >= 2:
    import scanpy as sc
    
    # Filter to cells with valid expression
    if 'within_threshold' in combined.obs.columns:
        combined_valid = combined[combined.obs['within_threshold']].copy()
    else:
        combined_valid = combined.copy()
    
    print(f'Valid cells: {combined_valid.n_obs:,}')
    print(f'Disease distribution:')
    print(combined_valid.obs['disease'].value_counts())
    
    # Basic preprocessing
    sc.pp.normalize_total(combined_valid)
    sc.pp.log1p(combined_valid)
    sc.pp.highly_variable_genes(combined_valid, n_top_genes=500, flavor='seurat_v3', span=0.3)
    
    # PCA and UMAP
    sc.pp.pca(combined_valid, n_comps=20)
    sc.pp.neighbors(combined_valid, n_neighbors=15)
    sc.tl.umap(combined_valid)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sc.pl.umap(combined_valid, color='disease', ax=axes[0], show=False, title='Disease')
    sc.pl.umap(combined_valid, color='sample_id', ax=axes[1], show=False, title='Sample')
    
    # Top differentially expressed genes
    sc.tl.rank_genes_groups(combined_valid, groupby='disease', method='wilcoxon')
    sc.pl.rank_genes_groups(combined_valid, ax=axes[2], show=False)
    
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'endo_vs_mla_preliminary.png'), dpi=150)
    plt.show()
    
    # Top DE genes
    print('\nTop differentially expressed genes (Endo vs MLA):')
    result = combined_valid.uns['rank_genes_groups']
    for group in result['names'].dtype.names:
        print(f'\n{group}:')
        for i in range(min(10, len(result['names'][group]))):
            gene = result['names'][group][i]
            score = result['scores'][group][i]
            pval = result['pvals_adj'][group][i]
            print(f'  {gene}: score={score:.3f}, padj={pval:.2e}')
else:
    print('Need at least 2 disease groups for comparison.')
    print('Process more samples and re-run this section.')

## 5b. Batch Protein-Gene Correlation Validation

For each processed sample, validate alignment by correlating COMET protein intensities
with their corresponding gene expression from the integrated AnnData.

In [ ]:
# Batch protein-gene correlation validation
all_corr_results = []

for adata in all_adatas:
    sid = adata.obs['sample_id'].iloc[0] if 'sample_id' in adata.obs.columns else 'unknown'
    disease = adata.obs['disease'].iloc[0] if 'disease' in adata.obs.columns else 'unknown'

    # Look for COMET protein parquet
    protein_path = Path(f'T:/Sammy Data/projects/out/out_comet/{sid}_BS_protein_combined.parquet')
    if not protein_path.exists():
        continue

    protein_df = pd.read_parquet(protein_path)
    n_min = min(len(protein_df), adata.n_obs)
    protein_df = protein_df.iloc[:n_min]

    corr = validate_protein_gene_correlation(adata, protein_df)
    if len(corr) > 0:
        corr['sample_id'] = sid
        corr['disease'] = disease
        all_corr_results.append(corr)

if all_corr_results:
    corr_all = pd.concat(all_corr_results, ignore_index=True)
    print(f'Protein-gene correlations across {len(all_corr_results)} samples:')
    print(corr_all.groupby(['protein', 'gene'])['spearman_rho'].agg(['mean', 'std', 'count']).round(3).to_string())

    # Save
    corr_path = OUTPUT_DIR / 'batch_protein_gene_correlations.csv'
    corr_all.to_csv(corr_path, index=False)
    print(f'\nSaved: {corr_path}')
else:
    print('No COMET protein data found for any processed sample.')

## 6. Final Summary

In [ ]:
print('=' * 70)
print('BATCH ALIGNMENT COMPLETE')
print('=' * 70)

# Count samples by status
total_endo_mla = len(ALL_ENDO_MLA)
processed = len(all_metrics) if 'all_metrics' in dir() else 0

print(f'\nTotal Endo/MLA samples: {total_endo_mla}')
print(f'  Endo: {len(ENDO_SAMPLES)}')
print(f'  MLA: {len(MLA_SAMPLES)}')
print(f'\nProcessed: {processed}')
print(f'Remaining: {total_endo_mla - processed}')

if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    print(f'\nAlignment Quality Summary:')
    for _, row in metrics_df.iterrows():
        print(f'  {row["sample_id"]} ({row["disease"]}): {row["quality"]} '
              f'(median={row["median_distance"]:.1f}px)')

print(f'\nOutput directory: {OUTPUT_DIR}')
print('=' * 70)